# Fase 0 - Ingesta de datos

In [3]:
import os
from urllib.request import urlretrieve

In [4]:
if not os.path.exists("../data"):
    os.makedirs("../data")

## Descargar Datasets Abiertos

![](../img/DownloadOpenData.png)

### Paso 1: Descargar datos

In [8]:
# source: https://data.imdb.com/non-commercial-datasets/

urlretrieve(
    "https://datasets.imdbws.com/title.basics.tsv.gz", 
    "../data/title.basics.tsv.gz"
)
    
urlretrieve(
    "https://datasets.imdbws.com/title.ratings.tsv.gz", 
    "../data/title.ratings.tsv.gz"
) 

('../data/title.ratings.tsv.gz', <http.client.HTTPMessage at 0xfffee3446ae0>)

In [1]:
!uv add polars

Resolved 3 packages in 0.88ms
Checked 2 packages in 0.06ms


In [2]:
import polars as pl
pl.__version__

'2.1.0'

### Paso 2: Lectura de datos

#### A) Basics Data

In [9]:
basics_df = pl.read_csv(
    "../data/title.basics.tsv.gz",
    separator="\t",
    null_values="\\N", # to handle missing values
    quote_char=None,   # to avoid issues with quotes in the data
)

In [10]:
basics_df.head()

tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
str,str,str,str,i64,i64,str,i64,str
"""tt0000001""","""short""","""Carmencita""","""Carmencita""",0,1894,null,1,"""Documentary,Short"""
"""tt0000002""","""short""","""Le clown et ses chiens""","""Le clown et ses chiens""",0,1892,null,5,"""Animation,Short"""
"""tt0000003""","""short""","""Poor Pierrot""","""Pauvre Pierrot""",0,1892,null,5,"""Animation,Comedy,Romance"""
"""tt0000004""","""short""","""Un bon bock""","""Un bon bock""",0,1892,null,12,"""Animation,Short"""
"""tt0000005""","""short""","""Blacksmith Scene""","""Blacksmith Scene""",0,1893,null,1,"""Short"""


In [11]:
# how many rows and columns are in the dataframe
basics_df.shape

(12902085, 9)

#### B) Ratings Data

In [13]:
ratings_df = pl.read_csv(
    "../data/title.ratings.tsv.gz",
    separator="\t",
    null_values="\\N", # to handle missing values
    quote_char=None,   # to avoid issues with quotes in the data
)
ratings_df.head()

tconst,averageRating,numVotes
str,f64,i64
"""tt0000001""",5.7,2238
"""tt0000002""",5.4,327
"""tt0000003""",6.4,2401
"""tt0000004""",5.0,203
"""tt0000005""",6.2,3104


In [14]:
ratings_df.shape

(1726141, 3)

#### C) Unificar Datasets

In [15]:
# merge the two dataframes on the tconst column, which is the unique identifier for each title in the IMDB dataset
movies_df = (
    basics_df
    .filter(pl.col("titleType") == "movie")  # filter for movies only
    .select([
        pl.col("tconst").alias("imdb_movie_id"), # rename the column to imdb_movie_id
        pl.col("primaryTitle").alias("title"), # rename the column to title
        pl.col("startYear").cast(pl.Int32).alias("year"), # rename the column to year and cast it to Int32
        pl.col("genres") # keep the genres column
    ])
    .join(
        ratings_df.select(["tconst", "averageRating", "numVotes"]), # select only the necessary columns from ratings_df
        left_on="imdb_movie_id",    # join on the imdb_movie_id column from basics_df
        right_on="tconst"           # join on the tconst column from ratings_df
    )
    .rename({       # rename columns in the merged dataframe
        "averageRating": "imdb_rating", # rename the column to imdb_rating
        "numVotes": "imdb_votes"        # rename the column to imdb_votes
    })
    .filter(
        (pl.col("imdb_votes") >= 1000)   # filter for movies with at least 1000 votes
    )
)

In [16]:
movies_df.head()

imdb_movie_id,title,year,genres,imdb_rating,imdb_votes
str,str,i32,str,f64,i64
"""tt0000574""","""The Story of the Kelly Gang""",1906,"""Action,Adventure,Biography""",6.0,1102
"""tt0002130""","""Dante's Inferno""",1911,"""Adventure,Drama,Fantasy""",7.1,4341
"""tt0002423""","""Passion""",1919,"""Biography,Drama,Romance""",6.7,1146
"""tt0002844""","""Fantômas: In the Shadow of the…",1913,"""Crime,Drama""",6.9,2778
"""tt0003014""","""Ingeborg Holm""",1913,"""Drama""",7.0,1642


In [17]:
movies_df.write_parquet(
    "../data/movies.parquet",
    compression="snappy", # use snappy compression for faster read/write
)

In [18]:
movies_df.shape

(49392, 6)

#### D) Guardar IDs de películas

In [24]:
# write the imdb_movie_id column to a text file, one id per line
with open("../data/movie_ids.txt", "w") as f:
    for movie_id in movies_df["imdb_movie_id"].to_list():
        f.write(f"{movie_id}\n")

In [25]:
!head ../data/movie_ids.txt

tt0000574
tt0002130
tt0002423
tt0002844
tt0003014
tt0003037
tt0003165
tt0003419
tt0003643
tt0003740


## Descargar Dataset Privado de OMDB API

![](../img/DownloadPrivateData.png)

### Paso A) Configuración

In [20]:
# Source: https://www.omdbapi.com/
!uv add dotenv loguru aiohttp orjson tqdm

Resolved 20 packages in 801ms                                        
Prepared 14 packages in 814ms                                                aiohttp              ------------------------------ 1.42 MiB/1.81 MiB           aiohttp              ------------------------------ 461.59 KiB/1.81 MiB         aiohttp              ------------------------------ 242.97 KiB/1.81 MiB         aiohttp              ------------------------------ 222.82 KiB/1.81 MiB         aiohttp              ------------------------------ 94.82 KiB/1.81 MiB          aiohttp              ------------------------------ 78.82 KiB/1.81 MiB          aiohttp              ------------------------------ 47.39 KiB/1.81 MiB          aiohttp              ------------------------------ 46.82 KiB/1.81 MiB          aiohttp              ------------------------------ 14.82 KiB/1.81 MiB          Aiohttp              ------------------------------     0 B/1.81 MiB            
Installed 14 packages in 10ms                       

In [21]:
from dotenv import load_dotenv
from loguru import logger
from tqdm import tqdm
from pathlib import Path

import aiohttp
import asyncio

import orjson

from typing import List, Dict

In [22]:
import os

# Cargar variables de entorno desde archivo .env (incluida la API KEY)
load_dotenv()

OMDB_API_KEY = os.getenv("OMDB_API_KEY")

In [26]:
# ========== CONFIGURACIÓN GLOBAL ==========
# URL base de la API de OMDb
OMDB_API_URL = "https://www.omdbapi.com/"
# Número de workers concurrentes para descargar datos en paralelo
# CONCURRENCY = 20
CONCURRENCY = 5
# Número máximo de reintentos por cada ID que falla
MAX_RETRIES = 5
# Ruta del archivo para guardar qué IDs ya fueron procesados
CHECKPOINT_FILE = "../data/processed_ids.txt"


# ========== CONFIGURACIÓN DE LOGURU ==========
# Eliminar los manejadores de logs por defecto
logger.remove()
# Agregar un nuevo manejador de logs con formato personalizado y colores
logger.add(
    lambda msg: tqdm.write(msg, end=""),  # Función que imprime el mensaje usando tqdm para no romper la barra de progreso
    # lambda msg: print(msg, end=""),  # Función que imprime el mensaje sin agregar salto de línea extra
    colorize=True,  # Habilitar colores en los logs
    format="<green>{time:HH:mm:ss}</green> | "  # Hora en verde
           "<level>{level}</level> | "  # Nivel de log (INFO, WARNING, etc.)
           "{message}\n",  # Mensaje del log
)

1

### Paso B: Checkpoint

In [27]:
# ========== UTILIDADES DE CHECKPOINT ==========
# Función para cargar IDs que ya fueron procesados
def load_processed_ids() -> set[str]:
    """Carga los IDs de IMDb ya procesados desde el archivo de checkpoint."""
    # Verificar si el archivo de checkpoint existe
    if Path(CHECKPOINT_FILE).exists():
        logger.info(f"Loading checkpoint from {CHECKPOINT_FILE}")
        # Abrir el archivo en modo lectura
        with open(CHECKPOINT_FILE, "r") as f:
            # Leer cada línea, eliminar espacios en blanco, y crear un conjunto
            return set(line.strip() for line in f if line.strip())
    # Si el archivo no existe, retornar un conjunto vacío
    return set()


# Función para guardar un ID procesado en el checkpoint
def append_processed_id(imdb_id: str) -> None:
    """Agrega un único ID de IMDb al archivo de checkpoint."""
    # Abrir el archivo en modo append (añadir al final)
    with open(CHECKPOINT_FILE, "a") as f:
        # Escribir el ID seguido de un salto de línea
        f.write(imdb_id + "\n")

### Paso C: Request a OMDB

In [28]:
# ========== FUNCIÓN PARA UN ÚNICO REQUEST A OMDb ==========
# Función asíncrona para obtener datos de una película de OMDb
async def fetch_one(session: aiohttp.ClientSession, imdb_id: str) -> Dict | None:
    """
    Obtiene un registro de OMDb por su ID de IMDb.
    Retorna: {"imdb_id": ..., "raw": <json_de_omdb>} o None si falla permanentemente.
    """
    # Definir los parámetros para el request a OMDb
    params = {
        "i": imdb_id,  # ID de IMDb (ej: tt1234567)
        "apikey": os.getenv("OMDB_API_KEY"),  # API key desde variables de entorno
        "plot": "full",  # Obtener la trama completa de la película
    }
    # Inicializar contador de reintentos
    retries = 0
    # logger.info(params)

    # Bucle que se ejecuta mientras no se alcance el máximo de reintentos
    while retries < MAX_RETRIES:
        try:
            # logger.info(f"[{imdb_id}] AFTER request")
            # Hacer un request GET asíncrono a la API
            async with session.get(OMDB_API_URL, params=params) as resp:
                # logger.info(f"[{imdb_id}] BEFORE request")
                # Obtener el código de estado HTTP
                status = resp.status

                # Manejar error 429: Too Many Requests (limite de rate)
                if status == 429:
                    # Calcular tiempo de espera: 5 segundos por reintento
                    wait = 5 * (retries + 1)
                    logger.warning(f"[{imdb_id}] 429 Too Many Requests — sleeping {wait}s")
                    # Esperar antes de reintentar
                    await asyncio.sleep(wait)
                    # Incrementar contador de reintentos
                    retries += 1
                    # Saltar a la siguiente iteración del bucle
                    continue

                # Manejar errores del servidor (5xx)
                if status >= 500:
                    # Calcular tiempo de espera: 3 segundos por reintento
                    wait = 3 * (retries + 1)
                    logger.warning(f"[{imdb_id}] Server error {status} — retrying in {wait}s")
                    # Esperar antes de reintentar
                    await asyncio.sleep(wait)
                    # Incrementar contador de reintentos
                    retries += 1
                    # Saltar a la siguiente iteración del bucle
                    continue

                # Convertir la respuesta a JSON
                data = await resp.json()

                # Verificar si OMDb retornó un error lógico (ID inválido, etc.)
                if data.get("Response") == "False":
                    # Obtener el mensaje de error de OMDb
                    err = data.get("Error", "Unknown error")
                    logger.error(f"[{imdb_id}] OMDb error: {err}")
                    # Retornar el resultado con el error (para guardar en log)
                    return {"imdb_id": imdb_id, "raw": data}

                # Si todo fue exitoso, retornar los datos (sin generar log para evitar spam)
                return {"imdb_id": imdb_id, "raw": data}

        # Manejar excepción de timeout
        except asyncio.TimeoutError:
            # Incrementar contador de reintentos
            retries += 1
            # Calcular tiempo de espera: 2 segundos por reintento
            wait = 2 * retries
            logger.warning(f"[{imdb_id}] Timeout — retrying in {wait}s")
            # Esperar antes de reintentar
            await asyncio.sleep(wait)

        # Manejar cualquier otra excepción inesperada
        except Exception as e:
            # Incrementar contador de reintentos
            retries += 1
            # Calcular tiempo de espera: 2 segundos por reintento
            wait = 2 * retries
            logger.error(f"[{imdb_id}] Unexpected error: {e} — retrying in {wait}s")
            # Esperar antes de reintentar
            await asyncio.sleep(wait)

    # Si se agotan todos los reintentos, registrar error crítico
    logger.critical(f"[{imdb_id}] FAILED permanently after {MAX_RETRIES} retries")
    # Retornar None para indicar fallo definitivo
    return None

### Paso D: Worker

In [29]:
# ========== WORKER: PROCESA IDs DE LA COLA Y ESCRIBE A JSONL ==========
# Función asíncrona que actúa como worker para procesar IDs en paralelo
async def worker(
    worker_id: int,  # Identificador del worker
    session: aiohttp.ClientSession,  # Sesión HTTP compartida
    queue: asyncio.Queue,  # Cola de IDs a procesar
    jsonl_path: Path,  # Ruta del archivo JSONL para guardar resultados
    write_lock: asyncio.Lock,  # Lock para escribir de forma segura en el archivo
    progress: tqdm # Barra de progreso opcional (para mostrar progreso en consola)
):
    # Bucle infinito que procesa IDs de la cola
    while True:
        # Obtener el siguiente ID de la cola (espera si está vacía)
        imdb_id = await queue.get()
        # Verificar si es la señal de terminación (None)
        if imdb_id is None:
            # Marcar la tarea como completada
            queue.task_done()
            # Salir del bucle para terminar el worker
            return

        # logger.info(f"Worker {worker_id}: {imdb_id}")

        # Obtener datos de OMDb para este ID
        result = await fetch_one(session, imdb_id)

        # Verificar si el fetch fue exitoso
        if result is not None:
            # Convertir el resultado a formato JSON y luego a string (usando orjson)
            line = orjson.dumps(result).decode("utf-8") + "\n"

            # Usar un lock para asegurar que solo un worker escriba a la vez
            async with write_lock:
                # Abrir el archivo JSONL en modo append
                with open(jsonl_path, "a", encoding="utf-8") as f:
                    # Escribir una línea JSON
                    f.write(line)

            # Guardar el ID como procesado en el checkpoint
            append_processed_id(imdb_id)
        
        # Actualizar la barra de progreso
        progress.update(1)

        # Marcar la tarea como completada en la cola
        queue.task_done()

### Paso E: Pipeline Asíncrono

In [30]:
# ========== FUNCIÓN PRINCIPAL: PIPELINE ASÍNCRONO ==========
# Función asíncrona principal que orquesta todo el descargador
async def main(
    all_ids: List[str],  # Lista de todos los IDs de IMDb a descargar
    output_file: str = "omdb_raw.jsonl",  # Ruta del archivo de salida JSONL
):
    """
    Punto de entrada principal del descargador de OMDb.
    
    - all_ids: lista de IDs de IMDb (formato: ttXXXXXX)
    - output_file: ruta del archivo JSONL para guardar los resultados
    
    Esta función:
    - Salta los IDs que ya fueron procesados
    - Descarga los IDs restantes con concurrencia
    - Guarda cada resultado como una línea JSON en output_file
    - Es segura para reintentar después de crashes
    """
    # Convertir el path a un objeto Path
    jsonl_path = Path(output_file)

    # Cargar los IDs que ya fueron procesados desde el checkpoint
    processed = load_processed_ids()
    # Crear una lista de IDs que aún no han sido procesados
    remaining = [i for i in all_ids if i not in processed]
    # Crear una barra de progreso para mostrar el avance de la descarga
    progress = tqdm(
        total=len(remaining),
        desc="Downloading OMDb",
        unit="movie",
        colour="green",
        dynamic_ncols=True,
    )

    # Registrar información sobre el progreso
    logger.info(f"Total IDs: {len(all_ids)}")
    logger.info(f"Already processed: {len(processed)}")
    logger.info(f"Remaining to fetch: {len(remaining)}")
    logger.info(f"Output JSONL: {jsonl_path}")

    # Si no hay IDs por procesar, salir de la función
    if not remaining:
        logger.success("Nothing to do. All IDs are already processed.")
        return

    # Crear una cola asíncrona para almacenar los IDs a procesar
    queue: asyncio.Queue = asyncio.Queue()
    # Crear un lock para sincronizar escrituras concurrentes
    write_lock = asyncio.Lock()

    # Encolar todos los IDs que falta procesar
    for imdb_id in remaining:
        await queue.put(imdb_id)

    # Configurar un timeout de 40 segundos para cada request HTTP
    timeout = aiohttp.ClientTimeout(total=40)

    # Crear una sesión HTTP asíncrona con el timeout configurado
    async with aiohttp.ClientSession(timeout=timeout) as session:
        # Crear y lanzar múltiples workers en paralelo
        workers = [
            asyncio.create_task(worker(i, session, queue, jsonl_path, write_lock, progress))
            for i in range(CONCURRENCY)
        ]

        # Encolar una señal de terminación (None) para cada worker
        for _ in workers:
            await queue.put(None)

        # Esperar hasta que todos los IDs de la cola hayan sido procesados
        await queue.join()

        # Cancelar cualquier worker que aún esté en ejecución
        for w in workers:
            w.cancel()

    # Cerrar la barra de progreso
    progress.close()

    # Registrar que la descarga completó exitosamente
    logger.success("Download complete.")
    logger.info(f"Results appended to {jsonl_path}")

#### Paso F: Ejecutar

In [ ]:
# ========== CARGAR LISTA DE IDs Y EJECUTAR EL DESCARGADOR ==========
with open("../data/movie_ids.txt") as f:
    # Leer todos los IDs de IMDb desde el archivo, eliminando espacios en blanco
    all_ids = [line.strip() for line in f if line.strip()]

# Run the async downloader
# await main(all_ids[:1500], output_file="../data/omdb_raw.jsonl")
await main(all_ids, output_file="../data/omdb_raw.jsonl")

16:59:16 | INFO | Total IDs: 1500

16:59:16 | INFO | Already processed: 0

16:59:16 | INFO | Remaining to fetch: 1500

16:59:16 | INFO | Output JSONL: ../data/omdb_raw.jsonl



17:00:04 | ERROR | [tt0030755] OMDb error: Request limit reached!

17:00:05 | ERROR | [tt0030811] OMDb error: Request limit reached!

17:00:05 | ERROR | [tt0030824] OMDb error: Request limit reached!

17:00:05 | ERROR | [tt0030764] OMDb error: Request limit reached!



17:00:05 | ERROR | [tt0030845] OMDb error: Request limit reached!

17:00:05 | ERROR | [tt0030848] OMDb error: Request limit reached!



17:00:05 | ERROR | [tt0030857] OMDb error: Request limit reached!

17:00:05 | ERROR | [tt0030865] OMDb error: Request limit reached!

17:00:05 | ERROR | [tt0030944] OMDb error: Request limit reached!

17:00:05 | ERROR | [tt0030879] OMDb error: Request limit reached!

17:00:05 | ERROR | [tt0030989] OMDb error: Request limit reached!



17:00:05 | ERROR | [tt0030993] OMDb error: Request limit reached!

17:00:05 | ERROR | [tt0030996] OMDb error: Request limit reached!

17:00:05 | ERROR | [tt0031002] OMDb error: Request limit reached!

17:00:05 | ERROR | [tt0031020] OMDb error: Request limit reached!

17:00:05 | ERROR | [tt0031022] OMDb error: Request limit reached!



...

17:00:27 | ERROR | [tt0035942] OMDb error: Request limit reached!

17:00:27 | ERROR | [tt0035951] OMDb error: Request limit reached!

17:00:27 | ERROR | [tt0035957] OMDb error: Request limit reached!

17:00:28 | ERROR | [tt0035958] OMDb error: Request limit reached!

17:00:28 | ERROR | [tt0035959] OMDb error: Request limit reached!



17:00:28 | ERROR | [tt0035962] OMDb error: Request limit reached!

17:00:28 | ERROR | [tt0035966] OMDb error: Request limit reached!

17:00:28 | ERROR | [tt0035979] OMDb error: Request limit reached!

17:00:28 | ERROR | [tt0036004] OMDb error: Request limit reached!

17:00:28 | ERROR | [tt0035980] OMDb error: Request limit reached!



17:00:28 | ERROR | [tt0036005] OMDb error: Request limit reached!

17:00:28 | ERROR | [tt0036009] OMDb error: Request limit reached!

17:00:28 | ERROR | [tt0036022] OMDb error: Request limit reached!

17:00:28 | ERROR | [tt0036027] OMDb error: Request limit reached!

17:00:28 | ERROR | [tt0036037] OMDb error: Request limit reached!



17:00:28 | ERROR | [tt0036038] OMDb error: Request limit reached!

17:00:28 | ERROR | [tt0036047] OMDb error: Request limit reached!

17:00:28 | ERROR | [tt0036055] OMDb error: Request limit reached!

17:00:28 | ERROR | [tt0036092] OMDb error: Request limit reached!

17:00:28 | ERROR | [tt0036094] OMDb error: Request limit reached!



17:00:28 | ERROR | [tt0036098] OMDb error: Request limit reached!

17:00:28 | SUCCESS | Download complete.

17:00:28 | INFO | Results appended to ../data/omdb_raw.jsonl

